In [1]:
import pandas as pd
import tqdm, os
import numpy as np
import cv2
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../..")

from utils.data_processing import dataset_df_to_token_df
from utils.scaling import fit_scalers, apply_scaler, describe
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aemri_articulatory_features

import torch
import torch.nn as nn
import torch.nn.functional as F

from utils.models import SequenceAutoencoder, VideoAutoencoder
from utils.train import (collate_video, collate_sequence, estimate_loss_scales, train_step,
                   latent_with_duration)

BATCH_SIZE = 32
MIN_FRAMES = 0
ALPHA = 0.5               # static vs velocity, meaningful because scales are fixed
DEVICE = "cpu"

In [2]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_aemri.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])
token_df = dataset_df_to_token_df(df)

In [3]:
token_df

,start,end,phone,art_f,ac_f,subject
0,17.34000,17.61000,s,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
1,12.63000,12.66000,ð,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
2,11.76000,11.83000,ʃ,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
3,11.23000,11.26000,b,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
4,8.86000,8.89000,æ,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
...,...,...,...,...,...,...
129379,21.78325,21.84325,ʌ,/Volumes/Transcend/USC-TIMIT/MRI/Data/M5/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M5/wav/u...,m5
129380,6.55325,6.58325,g,/Volumes/Transcend/USC-TIMIT/MRI/Data/M5/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M5/wav/u...,m5
129381,3.60325,3.67325,g,/Volumes/Transcend/USC-TIMIT/MRI/Data/M5/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M5/wav/u...,m5
129382,22.28325,22.37325,ɛ,/Volumes/Transcend/USC-TIMIT/MRI/Data/M5/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M5/wav/u...,m5


In [4]:
_REC_STATS = {}
 
 
def recording_stats(path, n_sample=400):
    """
    Pixel mean and sd over the WHOLE recording, cached per file.
 
    Note cv2's CAP_PROP_POS_FRAMES seeking on compressed video snaps to the
    nearest keyframe on some codecs. That is harmless for statistics, but it is
    worth checking separately that _read_video_segment returns frames that really
    correspond to the requested times — a silent half-second offset would
    misalign every token against the audio.
    """
    if path in _REC_STATS:
        return _REC_STATS[path]
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        cap.release()
        return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    idxs = np.unique(np.linspace(0, total - 1, min(n_sample, total)).astype(int))
    vals = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, fr = cap.read()
        if ok:
            vals.append(fr[:, :, 0].astype(np.float32))
    cap.release()
    if not vals:
        return None
    v = np.stack(vals)
    _REC_STATS[path] = (float(v.mean()), float(v.std()) + 1e-5)
    return _REC_STATS[path]
 
 
if os.path.isfile("scalers/aemri_video_stats.pkl"):
    with open("scalers/aemri_video_stats.pkl", "rb") as f:
        _REC_STATS = pickle.load(f)
else:
    for p in tqdm.tqdm(token_df["art_f"].unique(), desc="recording stats"):
        recording_stats(p)
    with open("scalers/aemri_video_stats.pkl", "wb") as f:
        pickle.dump(_REC_STATS, f)
print(f"intensity stats for {len(_REC_STATS)} recordings")

intensity stats for 849 recordings


In [5]:
# ==============================================================================
# Feature extraction, factored out so the scale pass and the train pass agree
# ==============================================================================

def token_features(row):
    stats = recording_stats(row["art_f"])
    art = get_aemri_articulatory_features(row["art_f"], row["start"], row["end"],
                                          stats=stats)
    if art is None or art.shape[0] < MIN_FRAMES:
        return None
    return torch.tensor(art, dtype=torch.float32), float(row["end"] - row["start"])


def iter_batches(df, batch_size=BATCH_SIZE, progress=None):
    """Yields (x, lengths, durations), padding to each batch's own max length."""
    buf = []
    rows = df.iterrows() if progress is None else progress
    for _, data in rows:
        item = token_features(data)
        if item is None:
            continue
        buf.append(item)
        if len(buf) == batch_size:
            yield collate_video(buf)
            buf = []
    if buf:                                   # the tail batch, previously dropped
        yield collate_video(buf)

In [6]:
# ==============================================================================
# Pass 0: fix the loss scales once, so ALPHA means something stable
# ==============================================================================

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

SCALES = estimate_loss_scales(
    ((x, L) for x, L, _ in iter_batches(shuffled_token_df.head(2000))),
    time_dim=2, max_batches=50)
print(f"loss scales: var(x)={SCALES[0]:.5f}  var(dx)={SCALES[1]:.5f}  "
      f"ratio {SCALES[0]/max(SCALES[1],1e-12):.1f}x")

loss scales: var(x)=0.98651  var(dx)=0.06806  ratio 14.5x


In [7]:
# ==============================================================================
# Config
# ==============================================================================

CKPT_DIR   = "ckpts/aemri"
CKPT_PATH  = f"{CKPT_DIR}/art_model_ckpt.pth"
LOG_PATH   = f"{CKPT_DIR}/art_model_training_log.csv"
SAVE_EVERY = 50
N_EPOCHS   = 3
LOG_KEYS   = ("total", "static", "delta", "duration")

# Set these only to force a resume point (e.g. recovering from an old-style
# checkpoint that stored a bare state_dict). Leave as None normally.
RESUME_EPOCH = None
RESUME_BATCH = None

os.makedirs(CKPT_DIR, exist_ok=True)

# ==============================================================================
# Model
# ==============================================================================

art_model = VideoAutoencoder(
    input_hw=(68, 68),
    latent_dim=64,
    input_channels=1,
    output_activation=None,
    stn_mode="shared",
    stn_transform="rigid",
).to(DEVICE)

art_optimizer = torch.optim.Adam(art_model.parameters(), lr=1e-3)

log = {"epoch": [], "batch_idx": [], **{k: [] for k in LOG_KEYS}}

# ==============================================================================
# Resume
# ==============================================================================

def save_ckpt(epoch, batch_idx):
    """epoch/batch_idx = the last COMPLETED step. Written atomically so an
    interrupt during saving can't leave a truncated checkpoint."""
    torch.save(
        {
            "epoch": epoch,
            "batch_idx": batch_idx,
            "model": art_model.state_dict(),
            "optimizer": art_optimizer.state_dict(),
        },
        CKPT_PATH + ".tmp",
    )
    os.replace(CKPT_PATH + ".tmp", CKPT_PATH)
    pd.DataFrame(log).to_csv(LOG_PATH, index=False)


start_epoch, start_batch, have_position = 1, 0, False

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)

    if isinstance(ckpt, dict) and "model" in ckpt:
        art_model.load_state_dict(ckpt["model"])
        art_optimizer.load_state_dict(ckpt["optimizer"])
        start_epoch, start_batch = ckpt["epoch"], ckpt["batch_idx"]
        have_position = True
        print(f"Resuming: epoch {start_epoch}, after batch {start_batch}")
    else:
        # Old format: bare state_dict, no optimizer state and no position.
        art_model.load_state_dict(ckpt)
        print("Loaded weights from an old-style checkpoint. "
              "Optimizer state is gone (expect a brief loss bump) and the "
              "position is unknown — set RESUME_EPOCH / RESUME_BATCH if you "
              "know where it stopped.")

    if RESUME_EPOCH is not None:
        start_epoch, start_batch = RESUME_EPOCH, (RESUME_BATCH or 0)
        have_position = True

    # Reload the log, dropping anything recorded after the checkpoint point
    # (those batches will be re-run, so keeping them would double-count).
    if os.path.exists(LOG_PATH):
        old = pd.read_csv(LOG_PATH)
        if "epoch" not in old.columns:
            old.insert(0, "epoch", start_epoch)
        if have_position:
            old = old[
                (old["epoch"] < start_epoch)
                | ((old["epoch"] == start_epoch) & (old["batch_idx"] <= start_batch))
            ]
        log = {k: old[k].tolist() for k in log}

# ==============================================================================
# Train
# ==============================================================================

art_model.train()

for epoch in range(start_epoch, N_EPOCHS + 1):
    # Same seed as the original run -> same shuffle -> skipping by batch index
    # lands on exactly the samples that were still pending.
    shuffled_token_df = token_df.sample(
        frac=1, random_state=42 + epoch
    ).reset_index(drop=True)

    skip = start_batch if epoch == start_epoch else 0

    pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

    for batch_idx, (x, lengths, durations) in enumerate(
            iter_batches(shuffled_token_df, progress=pbar), start=1):

        if batch_idx <= skip:
            continue

        x, lengths, durations = x.to(DEVICE), lengths.to(DEVICE), durations.to(DEVICE)

        stats = train_step(art_model, art_optimizer, x, lengths, durations,
                           alpha=ALPHA, scales=SCALES, duration_mode="off")

        log["epoch"].append(epoch)
        log["batch_idx"].append(batch_idx)
        for k in LOG_KEYS:
            log[k].append(stats[k])

        pbar.set_description(
            f"e{epoch} {batch_idx}: ART {stats['total']:.4f} "
            f"(s {stats['static']:.3f} d {stats['delta']:.3f})")

        if batch_idx % SAVE_EVERY == 0:
            save_ckpt(epoch, batch_idx)

    # End of epoch: next run should start clean at the following epoch.
    save_ckpt(epoch + 1, 0)

save_ckpt(N_EPOCHS + 1, 0)

e3 4043: ART 0.5109 (s 0.137 d 0.884): 100%|██████████| 129384/129384 [4:10:51<00:00,  8.60it/s] 


In [3]:
if os.path.isfile(f"scalers/aemri_ac.pkl"):
    with open(f"scalers/aemri_ac.pkl", "rb") as f:
        scalers = pickle.load(f)
else:
    def raw_features(row):
        m = get_mfccs(
                row["ac_f"], row["start"], row["end"])
        if m is None or m.shape[0] < MIN_FRAMES:
            return None
        return m

    scalers = fit_scalers(token_df, raw_features, mode="per_feature")
    describe(scalers, name="MRI ")

    with open(f"scalers/aemri_ac.pkl", "wb") as f:
        pickle.dump(scalers, f)

In [4]:
# ==============================================================================
# Feature extraction, factored out so the scale pass and the train pass agree
# ==============================================================================

def token_features(data, scalers):
    ac = get_mfccs(data["ac_f"], data["start"], data["end"])
    if ac is None or ac.shape[0] < MIN_FRAMES:
        return None
    ac = apply_scaler(ac, scalers[data["subject"]])
    return torch.tensor(ac, dtype=torch.float32), float(data["end"] - data["start"])

def iter_batches(df, scalers, batch_size=BATCH_SIZE, progress=None):
    """Yields (x, lengths, durations), padding to each batch's own max length."""
    buf = []
    rows = df.iterrows() if progress is None else progress
    for _, data in rows:
        item = token_features(data, scalers)
        if item is None:
            continue
        buf.append(item)
        if len(buf) == batch_size:
            yield collate_sequence(buf)
            buf = []
    if buf:                                   # the tail batch, previously dropped
        yield collate_sequence(buf)

In [5]:
# ==============================================================================
# Pass 0: fix the loss scales once, so ALPHA means something stable
# ==============================================================================

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

SCALES = estimate_loss_scales(
    ((x, L) for x, L, _ in iter_batches(shuffled_token_df.head(2000), scalers)),
    time_dim=1, max_batches=50)
print(f"loss scales: var(x)={SCALES[0]:.5f}  var(dx)={SCALES[1]:.5f}  "
      f"ratio {SCALES[0]/max(SCALES[1],1e-12):.1f}x")

loss scales: var(x)=1.01686  var(dx)=0.30557  ratio 3.3x


In [6]:
N_EPOCHS   = 5

In [7]:
# ==============================================================================
# Model
# ==============================================================================

ac_model = SequenceAutoencoder(
    feature_shape=13,
    latent_dim=64,        # was 1024. With ~30k tokens that was memorisation
                          # territory, and Euclidean distances concentrate badly
                          # in 1024-d, which is bad for what you use z for.
).to(DEVICE)

ac_optimizer = torch.optim.Adam(ac_model.parameters(), lr=1e-3)

log = {"batch_idx": [], "total": [], "static": [], "delta": [], "duration": []}

# ==============================================================================
# Train
# ==============================================================================

ac_model.train()

for epoch in range(1, N_EPOCHS + 1):
    shuffled_token_df = token_df.sample(frac=1, random_state=42+epoch).reset_index(drop=True)
    pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

    for batch_idx, (x, lengths, durations) in enumerate(
            iter_batches(shuffled_token_df, scalers, progress=pbar), start=1):

        x, lengths, durations = x.to(DEVICE), lengths.to(DEVICE), durations.to(DEVICE)

        # time_dim=1 is inferred from the model class; masking, gradient clipping and
        # the detached duration probe all live in here.
        stats = train_step(ac_model, ac_optimizer, x, lengths, durations,
                        alpha=ALPHA, scales=SCALES, duration_mode="off")

        log["batch_idx"].append(batch_idx)
        for k in ("total", "static", "delta", "duration"):
            log[k].append(stats[k])
        pbar.set_description(
            f"{batch_idx}: AC {stats['total']:.4f} "
            f"(s {stats['static']:.3f} d {stats['delta']:.3f})")

        if batch_idx % 300 == 0:
            torch.save(ac_model.state_dict(), "ckpts/aemri/ac_model_ckpt.pth")
            pd.DataFrame(log).to_csv("ckpts/aemri/ac_model_training_log.csv", index=False)

torch.save(ac_model.state_dict(), "ckpts/aemri/ac_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/aemri/ac_model_training_log.csv", index=False)

4041: ART 0.3219 (s 0.172 d 0.471): 100%|██████████| 129384/129384 [14:48<00:00, 145.62it/s]
